In [1]:
!pip install boto3


   ---------------------------------------- 0.0/139.4 kB ? eta -:--:--
   -- ------------------------------------- 10.2/139.4 kB ? eta -:--:--
   -- ------------------------------------- 10.2/139.4 kB ? eta -:--:--
   -------- ------------------------------ 30.7/139.4 kB 259.2 kB/s eta 0:00:01
   ----------- --------------------------- 41.0/139.4 kB 245.8 kB/s eta 0:00:01
   ----------------- --------------------- 61.4/139.4 kB 272.3 kB/s eta 0:00:01
   -------------------- ------------------ 71.7/139.4 kB 245.8 kB/s eta 0:00:01
   -------------------------------------- 139.4/139.4 kB 434.7 kB/s eta 0:00:00
   ---------------------------------------- 0.0/14.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.1 MB ? eta -:--:--
   ---------------------------------------- 0.1/14.1 MB 825.8 kB/s eta 0:00:18
   ---------------------------------------- 0.1/14.1 MB 563.7 kB/s eta 0:00:25
   ---------------------------------------- 0.1/14.1 MB 656.4 kB/s eta 0:00:22
   -----


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\guada\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import os
import boto3

In [3]:
os.environ['S3_ENDPOINT_URL'] = 'http://minio:9000'
os.environ['AWS_ACCESS_KEY_ID'] = 'AKFBKGE170596'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'SKFBKGE200301'

In [ ]:
s3 = boto3.client(
    's3',
    endpoint_url=os.environ['S3_ENDPOINT_URL'],
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY']
)

bucket_name = 'dataspace'
path_dtm = 'city-data/dtm/latest/'

In [8]:
from rasterio.merge import merge

dtm_list = s3.list_objects_v2(Bucket=bucket_name, Prefix=path_dtm)['Contents']
# --- Load all DTM tiles ---
dtm_sources = []
for obj in dtm_list:
    filename = obj['Key'].split('/')[-1]
    if not filename.endswith('.ASC'):
        continue

    dtm_data = s3.get_object(Bucket=bucket_name, Key=path_dtm + filename)
    src = rasterio.open(io.BytesIO(dtm_data['Body'].read()))
    dtm_sources.append(src)

# --- Merge ---
mosaic_array, mosaic_transform = merge(dtm_sources)

# --- Update profile ---
profile = dtm_sources[0].profile
profile.update(
    height=mosaic_array.shape[1],
    width=mosaic_array.shape[2],
    transform=mosaic_transform,
    compress='lzw'
)

# --- Save mosaic ---
out_path = 'dtm_mosaic.tif'
with rasterio.open(out_path, 'w', **profile) as dst:
    dst.write(mosaic_array)

# --- Close sources ---
for src in dtm_sources:
    src.close()


EndpointConnectionError: Could not connect to the endpoint URL: "http://minio:9000/dataspace?list-type=2&prefix=city-data%2Fdtm%2Flatest%2F&encoding-type=url"